# BioKEA variable-aperture cassette: exact-byte and protocol verification

**Virtual CAD + publication-custody protocol only.** No specimens, fluids, robots or sequencing were run.

In [1]:
from pathlib import Path
import json, hashlib, subprocess, tempfile, sys
R=Path.cwd();assert (R/'variable_aperture_cassette.scad').is_file()
sys.path.insert(0,str(R))
from lomo_plan import enumerate_design
from custody_watch import snapshot
print('CASSETTE SOURCE SHA256',hashlib.sha256((R/'variable_aperture_cassette.scad').read_bytes()).hexdigest())

CASSETTE SOURCE SHA256 d5b37cc2cd9a8c5d6622a5755a04d356469acff3ad93f291b632f11fff564453


## Deterministic nominal CAD geometry (not fluidic fit)

In [2]:
c=json.loads((R/'module_catalog.json').read_text());p=c['aperture']
for offset,declared in zip(p['shift_mm'],p['open_width_mm']):
 observed=p['nominal_slot_width_mm']-offset
 assert abs(observed-declared)<1e-12
 print({'offset_mm':offset,'nominal_open_width_mm':observed,'observed_wet_throughput':'NOT_TESTED'})
assert p['bounds']=='ONLY_EMPTY_AND_DRY_ADJUSTMENT'

{'offset_mm': 0, 'nominal_open_width_mm': 3.6, 'observed_wet_throughput': 'NOT_TESTED'}
{'offset_mm': 1.2, 'nominal_open_width_mm': 2.4000000000000004, 'observed_wet_throughput': 'NOT_TESTED'}
{'offset_mm': 2.4, 'nominal_open_width_mm': 1.2000000000000002, 'observed_wet_throughput': 'NOT_TESTED'}


## Exact-byte source references and vendor DXF exports

In [3]:
s=json.loads((R/'source_inventory.json').read_text())
for src in s['source_repositories']:
 assert len(src['head'])==40
 for f in src['files']:
  assert f['status']=='EXACT_BYTES_READ' and len(f['sha256'])==64
 print(src['repo'],src['head'],len(src['files']),'files inventoried')
for name in ('base','slider','retainer'):
 a=R/'output'/f'variable_flat_{name}.dxf'
 assert a.read_bytes().startswith(b'999\nDXF from OpenSCAD')
 print(a.name,'SHA256',hashlib.sha256(a.read_bytes()).hexdigest())

BioKEA/BugSorter a2e3e1c7b151d5ea6db40dc1974f0c4159d7d634 3 files inventoried
zlite/entosieve a2e3e1c7b151d5ea6db40dc1974f0c4159d7d634 3 files inventoried
BioKEA/BugPicker 1ef0b97e63998ef492adc2c122b7d2296be01ad2 6 files inventoried
BioKEA/BugPicker-arm-demo bfd3be6ac8b5092f63824ccc656a3abb008a4f4a 4 files inventoried
huggingface/lerobot d3793b7e36522321c1c455f0dc4826db3793dceb 3 files inventoried
variable_flat_base.dxf SHA256 1ae8fe3267a5d2f66220198cd2eab0d6a070f5615c75219a0e4ed4316624c9bc
variable_flat_slider.dxf SHA256 1ae8fe3267a5d2f66220198cd2eab0d6a070f5615c75219a0e4ed4316624c9bc
variable_flat_retainer.dxf SHA256 c5366d97075da9e5be0f24e08246216096116deac82b5916e55371b4c32e0d0d


## Proposed LOMO, explicitly null performance

In [4]:
design=enumerate_design(c)
assert len(design['rows'])==18
assert all(row['comparison_result']=='NOT_TESTED' and row['delta_capture'] is None for row in design['rows'])
print(design['protocol_id'],len(design['rows']),'planned contrasts; outcomes NOT_COMPUTED')

biokea-lomo-sha256-0e3165df2ef512ae1f222eefeaf072d2a1a3d6b00035922545571b060f69da8a 18 planned contrasts; outcomes NOT_COMPUTED


## Read-only predecessor MMR recomputation; not altered

In [5]:
import importlib.util
source=R.parent/'biokea_scad_custody_20261025'
sys.path.insert(0,str(source))
import cad_build as b
v=b.verify();assert v['PASS']
print('PARENT',v['leaves'],v['root'],v['signing'])

PARENT 18 753c271e4672faf6294bd5c9b803a0ac77f53bdbd09442345e8a28b5e43ea2df NOT_SIGNED


## Tamper rejection in isolated temporary copy (no actual file modified)

In [6]:
assert subprocess.run([sys.executable,'custody_watch.py','self-test'],check=True,capture_output=True,text=True).returncode==0
print('INDEPENDENT_NEGATIVE_TEST_PASS (temporary files only)')

INDEPENDENT_NEGATIVE_TEST_PASS (temporary files only)


## Interpretation limits

Image identity ≠ specimen identity. Merkle identity ≠ scientific truth. Laser-cut patterns need vendor tolerance and sample-safety qualification. Physical recovery, heat, chemical exposure, microscopy accuracy and taxonomic novelty remain NOT_TESTED.